# Local OCR & PDF Text Extraction Sandbox
This notebook evaluates different local libraries for extracting text from PDF files. Since these documents contain sensitive information, all models and processing pipelines run locally.

## Evaluated Methods:
1. **PyMuPDF (fitz)**: High-speed, robust C-based library to extract digital text layers.
2. **pdfplumber**: Excellent for digital text layers and table layout structure preservation.
3. **pypdf**: Lightweight, pure-python digital text extractor.
4. **pymupdf4llm**: Specialized tool to output layout-aware Markdown format (including tables).
5. **EasyOCR**: Local Deep Learning OCR model (CRAFT detector + CRNN recognizer) to extract text from scanned pages/images.

## 1. Import Libraries

In [1]:
import os
import time
import pandas as pd
import numpy as np
import fitz  # PyMuPDF
import pdfplumber
import pypdf
import pymupdf4llm
import easyocr

print("All libraries imported successfully!")

All libraries imported successfully!


## 2. Configuration & Paths

In [2]:
PDF_PATH = "files/lpb_oficios_01_06_26/8_operadora_y_desarrolladora_de_industrias.pdf"
print(f"Target PDF Path: {PDF_PATH}")
print(f"File exists: {os.path.exists(PDF_PATH)}")
if os.path.exists(PDF_PATH):
    print(f"File size: {os.path.getsize(PDF_PATH) / 1024:.2f} KB")

Target PDF Path: files/lpb_oficios_01_06_26/8_operadora_y_desarrolladora_de_industrias.pdf
File exists: True
File size: 996.31 KB


## 3. Check PDF Properties
Let's check if the PDF has a selectable text layer (digital) or if it is purely an image (scanned).

In [3]:
def analyze_pdf_type(pdf_path):
    doc = fitz.open(pdf_path)
    total_chars = 0
    pages_with_text = 0
    for page_num, page in enumerate(doc):
        text = page.get_text()
        char_count = len(text.strip())
        total_chars += char_count
        if char_count > 0:
            pages_with_text += 1
            
    print(f"Total pages: {len(doc)}")
    print(f"Pages with selectable text: {pages_with_text}")
    print(f"Total character count: {total_chars}")
    
    if total_chars > 100:
        print("Conclusion: This is a DIGITAL PDF with a selectable text layer.")
    else:
        print("Conclusion: This is a SCANNED PDF (images). OCR is required to extract text.")
        
analyze_pdf_type(PDF_PATH)

Total pages: 2
Pages with selectable text: 0
Total character count: 0
Conclusion: This is a SCANNED PDF (images). OCR is required to extract text.


## 4. Define Extraction Functions

In [4]:
# 1. PyMuPDF Plain Text
def extract_pymupdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page_num, page in enumerate(doc):
        text += f"--- Page {page_num + 1} ---\n"
        text += page.get_text() + "\n"
    return text

# 2. pdfplumber Plain Text
def extract_pdfplumber(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            text += f"--- Page {page_num + 1} ---\n"
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

# 3. pypdf Plain Text
def extract_pypdf(pdf_path):
    reader = pypdf.PdfReader(pdf_path)
    text = ""
    for page_num, page in enumerate(reader.pages):
        text += f"--- Page {page_num + 1} ---\n"
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

# 4. pymupdf4llm Markdown with EasyOCR plugin
def easyocr_plugin(page, pixmap=None, dpi=150, language=None, **kwargs):
    global _easyocr_reader_global
    if '_easyocr_reader_global' not in globals():
        _easyocr_reader_global = easyocr.Reader(['es', 'en'], gpu=False)
    if pixmap is None:
        pixmap = page.get_pixmap(dpi=dpi)
    img_data = np.frombuffer(pixmap.samples, dtype=np.uint8).reshape(pixmap.h, pixmap.w, pixmap.n)
    if pixmap.n == 4:
        import cv2
        img_data = cv2.cvtColor(img_data, cv2.COLOR_RGBA2RGB)
    elif pixmap.n == 1:
        import cv2
        img_data = cv2.cvtColor(img_data, cv2.COLOR_GRAY2RGB)
    results = _easyocr_reader_global.readtext(img_data)
    scale = 72.0 / dpi
    for bbox, text, prob in results:
        x0, y0 = bbox[0]
        x2, y2 = bbox[2]
        rect = fitz.Rect(x0 * scale, y0 * scale, x2 * scale, y2 * scale)
        try:
            page.insert_text(rect.tl, text, fontsize=9)
        except:
            pass

def extract_pymupdf4llm(pdf_path):
    doc = fitz.open(pdf_path)
    total_chars = sum([len(page.get_text().strip()) for page in doc])
    if total_chars > 100:
        return pymupdf4llm.to_markdown(pdf_path)
    else:
        return pymupdf4llm.to_markdown(pdf_path, ocr_function=easyocr_plugin, force_ocr=True, dpi=150)

# 5. EasyOCR (scanned pages fallback)
def extract_easyocr(pdf_path):
    reader = easyocr.Reader(['es', 'en'], gpu=False)
    doc = fitz.open(pdf_path)
    text = ""
    for page_num, page in enumerate(doc):
        text += f"--- Page {page_num + 1} ---\n"
        pix = page.get_pixmap(dpi=150)
        img_data = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
        
        if pix.n == 4:
            import cv2
            img_data = cv2.cvtColor(img_data, cv2.COLOR_RGBA2RGB)
            
        results = reader.readtext(img_data)
        page_text = "\n".join([item[1] for item in results])
        text += page_text + "\n"
    return text

## 5. Evaluate and Compare Models
Let's run each extraction method, measure execution time, and check extraction results.

In [5]:
results = {}

# Test PyMuPDF
print("Running PyMuPDF...")
start = time.time()
try:
    text_pymupdf = extract_pymupdf(PDF_PATH)
    results["PyMuPDF"] = {
        "time": time.time() - start,
        "char_count": len(text_pymupdf),
        "text": text_pymupdf,
        "success": True,
        "output_type": "Plain Text"
    }
except Exception as e:
    results["PyMuPDF"] = {"time": 0, "char_count": 0, "text": "", "success": False, "error": str(e), "output_type": "Plain Text"}

# Test pdfplumber
print("Running pdfplumber...")
start = time.time()
try:
    text_pdfplumber = extract_pdfplumber(PDF_PATH)
    results["pdfplumber"] = {
        "time": time.time() - start,
        "char_count": len(text_pdfplumber),
        "text": text_pdfplumber,
        "success": True,
        "output_type": "Plain Text"
    }
except Exception as e:
    results["pdfplumber"] = {"time": 0, "char_count": 0, "text": "", "success": False, "error": str(e), "output_type": "Plain Text"}

# Test pypdf
print("Running pypdf...")
start = time.time()
try:
    text_pypdf = extract_pypdf(PDF_PATH)
    results["pypdf"] = {
        "time": time.time() - start,
        "char_count": len(text_pypdf),
        "text": text_pypdf,
        "success": True,
        "output_type": "Plain Text"
    }
except Exception as e:
    results["pypdf"] = {"time": 0, "char_count": 0, "text": "", "success": False, "error": str(e), "output_type": "Plain Text"}

# Test pymupdf4llm
print("Running PyMuPDF4LLM...")
start = time.time()
try:
    text_pymupdf4llm = extract_pymupdf4llm(PDF_PATH)
    results["PyMuPDF4LLM"] = {
        "time": time.time() - start,
        "char_count": len(text_pymupdf4llm),
        "text": text_pymupdf4llm,
        "success": True,
        "output_type": "Markdown"
    }
except Exception as e:
    results["PyMuPDF4LLM"] = {"time": 0, "char_count": 0, "text": "", "success": False, "error": str(e), "output_type": "Markdown"}

# Test EasyOCR
print("Running EasyOCR (scanned fallback)...")
start = time.time()
try:
    text_easyocr = extract_easyocr(PDF_PATH)
    results["EasyOCR"] = {
        "time": time.time() - start,
        "char_count": len(text_easyocr),
        "text": text_easyocr,
        "success": True,
        "output_type": "Plain Text"
    }
except Exception as e:
    results["EasyOCR"] = {"time": 0, "char_count": 0, "text": "", "success": False, "error": str(e), "output_type": "Plain Text"}

print("All models executed!")

Running PyMuPDF...
Running pdfplumber...


Using CPU. Note: This module is much faster with a GPU.


Running pypdf...
Running PyMuPDF4LLM...


/home/lenovo/Documents/projects/ocr-uif/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


=== Document parser messages ===


OCR on page.number=0/1.
OCR on page.number=1/2.



Using CPU. Note: This module is much faster with a GPU.


Running EasyOCR (scanned fallback)...


All models executed!


## 6. Comparison Results

In [6]:
df_comparison = pd.DataFrame([
    {
        "Method": name,
        "Execution Time (s)": f"{data['time']:.4f}" if data['success'] else "Failed",
        "Character Count": data['char_count'] if data['success'] else 0,
        "Output Format": data['output_type'],
        "Status": "Success" if data['success'] else f"Failed: {data.get('error')}"
    }
    for name, data in results.items()
])
display(df_comparison)

,Method,Execution Time (s),Character Count,Output Format,Status
0,PyMuPDF,0.0199,32,Plain Text,Success
1,pdfplumber,0.0097,30,Plain Text,Success
2,pypdf,0.0049,30,Plain Text,Success
3,PyMuPDF4LLM,117.8941,5152,Markdown,Success
4,EasyOCR,61.0182,4937,Plain Text,Success


## 7. Sample Outputs (Snippets)

In [7]:
for name, data in results.items():
    if data['success']:
        print(f"\n=====================================")
        print(f" METHOD: {name} (Snippet)")
        print(f"=====================================")
        snippet = data['text'][:800]
        print(snippet)
        if len(data['text']) > 800:
            print("...\n[TRUNCATED]")


 METHOD: PyMuPDF (Snippet)
--- Page 1 ---

--- Page 2 ---



 METHOD: pdfplumber (Snippet)
--- Page 1 ---
--- Page 2 ---


 METHOD: pypdf (Snippet)
--- Page 1 ---
--- Page 2 ---


 METHOD: PyMuPDF4LLM (Snippet)
Hacienda 

UUF UNIDAD DE FTALICENCIA 

Secretaría de Haclenda y Crédito Público 

MÉXICO 

Este documento contiene información RESERVADA y CONFIDENCIAL de conformidad con lo establecido en el artículo 112, fracciones | IV; VII y XVII;y 715 de laGeneral de Transparencia y Acceso a la Información Pública, porLey lo que no deberá darse a conocer su contenido 

Dirección General de Integración de la Lista de Personas Bloqueadas y Procedimientos de Garantías de Audiencia Oficio No. 110/G/1329/2026 Ciudad de México; a 26 de mayo de 2026 

Mtro. Juan Ayax Fuentes Mendoza Vicepresidente de Supervisión de Procesos Preventivos, de la Comisión Nacional Bancaria y de Valores. Presente 

Conforme a las atribuciones previstas en la fracción XXXIV , del artículo 10 y; V, del artículo 10-G, de

## 8. Save MVP Deliverables

In [8]:
# Select the best plain text content
is_digital = results["PyMuPDF"]["char_count"] > 100
best_plain_method = "PyMuPDF" if is_digital else "EasyOCR"
plain_text_out = results[best_plain_method]["text"]

# Save MVP 1
with open("8_operadora_y_desarrolladora_de_industrias_plain.txt", "w", encoding="utf-8") as f:
    f.write(plain_text_out)
print(f"Saved MVP 1 (Plain Text) to '8_operadora_y_desarrolladora_de_industrias_plain.txt'")

# Save MVP 2
best_md_method = "PyMuPDF4LLM"
md_text_out = results[best_md_method]["text"]
with open("8_operadora_y_desarrolladora_de_industrias_markdown.md", "w", encoding="utf-8") as f:
    f.write(md_text_out)
print(f"Saved MVP 2 (Markdown) to '8_operadora_y_desarrolladora_de_industrias_markdown.md'")

Saved MVP 1 (Plain Text) to '8_operadora_y_desarrolladora_de_industrias_plain.txt'
Saved MVP 2 (Markdown) to '8_operadora_y_desarrolladora_de_industrias_markdown.md'
